# Exploratory Data Analysis (EDA) - E-commerce Dataset

This notebook performs exploratory data analysis on the SQLite database containing `Customers`, `Products`, and `Orders` tables.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic style
sns.set_theme(style="darkgrid", palette="deep")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Connect to the database
conn = sqlite3.connect('ecommerce.db')

# Read data into DataFrames
df_orders = pd.read_sql_query("SELECT * FROM Orders", conn)
df_customers = pd.read_sql_query("SELECT * FROM Customers", conn)
df_products = pd.read_sql_query("SELECT * FROM Products", conn)

df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

### 1. Revenue by Category

In [ ]:
query = """
SELECT p.category, SUM(o.total_amount) as revenue
FROM Products p
JOIN Orders o ON p.product_id = o.product_id
GROUP BY p.category
ORDER BY revenue DESC
"""
df_revenue_category = pd.read_sql_query(query, conn)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_revenue_category, x='category', y='revenue', palette='viridis')
plt.title('Total Revenue by Product Category', fontsize=16)
plt.xlabel('Category')
plt.ylabel('Revenue ($)')
plt.show()

### 2. Monthly Sales Trend

In [ ]:
df_orders['month_year'] = df_orders['order_date'].dt.to_period('M')
monthly_sales = df_orders.groupby('month_year')['total_amount'].sum().reset_index()
monthly_sales['month_year'] = monthly_sales['month_year'].astype(str)

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_sales, x='month_year', y='total_amount', marker='o', color='crimson')
plt.title('Monthly Sales Trend', fontsize=16)
plt.xticks(rotation=45)
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.show()

### 3. Customer Distribution by Country

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df_customers, y='country', order=df_customers['country'].value_counts().index, palette='mako')
plt.title('Customer Distribution by Country', fontsize=16)
plt.xlabel('Number of Customers')
plt.ylabel('Country')
plt.show()

In [ ]:
conn.close()